In [ ]:
#importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Base Model: Linnear Regression, Random Forest & Gradient boosting
We create 3 models with different algorythms to compare,algorythm is the way the model will learn the data and predict an output after. Chosing a good algoryth is crucial because it will directly impact the efficiency model.
We will test a linnear regression algorythm, a random forest algorythm and a Gradiant boosting algorythm.
**Linear-regression** models are relatively simple and provide an easy-to-interpret mathematical formula that can generate predictions.
**Random-forest** models combines the output of multiple decision trees to reach a single result
**Gradiant Boosting** is a machine learning technique that combines multiple weak prediction models into a single
ensemble
## Step 1: Loading the Data
We import our cleaned_data with all data needed

In [ ]:
# We load the cleaned dataset and precise that each column is separated by a semicolon
df  = pd.read_csv('cleaned_dataset.csv', sep=';')

## Step 2: Defining the Goal (Target) and what is usable (Features)
To train an AI, we must tell it what it needs to guess (the **Target**) and what information it is allowed to use (the **Features**). 
A model will then try to find correlations between the target and the features to build his algorythm.

Then, we clean the table to remove every rows where the value is missing because the Ai can't exploit lines where one value is missing.

In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We remove rows where the delay is equal to -1 and nan (meaning the data was invalid during cleaning).
df = df[df[target] != -1]
df = df.dropna()

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 3: Translating Text for the Computer (Encoding)
An Artificial Intelligence is a calculating machine: it only understands mathematics. It cannot read words like "Paris" or "Bordeaux". 
We must therefore use a "translator" to convert these station names into numerical codes that the computer can analyze. Encoding of columns where the value isn't usable by the model (strings) into and usable value.
We call it the preprocessor.

In [ ]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=False is used to return a dense array instead of a sparse matrix, which is easier to handle in pipelines.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    # passthrough means that the columns that are not specified in the transformers list (in this case, the numerical features) will be passed through without any transformation.
    remainder='passthrough'
)

## Step 4: Creating the AI "Assembly Line"
We are going to set up our predictions models.
We setup a Linear Regression model, a Random Forest model and a gradient boosting model
We send each models trough the *preprocessor* to clear unreadable values and then we setup the real algorythm through the *regressor*
Then we use a *pipeline*, which is a serie of instruction that the data will go through. In our case it ll go through the encoder with the preprocessor and then through the model algorythm which is the regressor.

In [ ]:
# We create a "Pipeline": it is an automated assembly line.
# We create two pipelines: one for the linear regression model and one for the random forest model.
# Linear Regression:
linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])
# Random Forest:
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a random forest algorithm.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])
# Gradient Boosting:
gradient_boosting_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

## Step 5: Creating the final AI
This is the most important step. We will divide our data into two batches:
- **80% for training:** The AI practices guessing the delays and looks at the real answers to learn from its mistakes.
- **20% for testing:** We hide the answers from the AI and ask it to make its predictions to see if it has understood the logic.

And then we train both of our models based on the exact same settings to see which is better.
We also setup a dummy model that will serve as a baseline to see if our models are really an improvement. The baseline is based on the mean.

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
# We train both models to compare their performance later.
linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)
gradient_boosting_model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred_lin = linear_model.predict(X_test)
y_pred_forest = forest_model.predict(X_test)
y_pred_gradient_boosting = gradient_boosting_model.predict(X_test)
# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 6: Grading (Performance Evaluation)
Now that the AI has taken its exam and made its predictions, we will compare its answers with reality to give it performance grades.
Then we will compare the result of all of our models to see the better one and improve it later on.
For the comparison we use 3 metrics :
    - Mean Absolute Error : by how many minutes was the model wrong
    - Mean Squared Error : Basically the same as MAE except that errors are heavily punished
    - R² : The metrics used to see the performance of the model
The we compare the R² with cross-validation to see if the model perform well in any situation or if the previous values were pure luck.

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : Linear Regression: {mean_absolute_error(y_test, y_pred_lin):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Random Forest: {mean_absolute_error(y_test, y_pred_forest):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Gradient Boosting: {mean_absolute_error(y_test, y_pred_gradient_boosting):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : Linear Regression: {mean_squared_error(y_test, y_pred_lin):.2f}")
print(f"Mean Squared Error (MSE) : Random Forest: {mean_squared_error(y_test, y_pred_forest):.2f}")
print(f"Mean Squared Error (MSE) : Gradient Boosting: {mean_squared_error(y_test, y_pred_gradient_boosting):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : Linear Regression: {r2_score(y_test, y_pred_lin):.2f}")
print(f"R² Score : Random Forest: {r2_score(y_test, y_pred_forest):.2f}")
print(f"R² Score : Gradient Boosting: {r2_score(y_test, y_pred_gradient_boosting):.2f}")
# We also use a cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_linear_r2 = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_forest_r2 = cross_val_score(forest_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_gradient_boosting_r2 = cross_val_score(gradient_boosting_model, X, y, cv=5, scoring='r2', n_jobs=-1)
score_baseline_r2 = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"\nR² average with Cross-Validation 5 : Linear Regression: {np.mean(scores_linear_r2):.2f} (+/- {np.std(scores_linear_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Random Forest: {np.mean(scores_forest_r2):.2f} (+/- {np.std(scores_forest_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Gradient Boosting: {np.mean(scores_gradient_boosting_r2):.2f} (+/- {np.std(scores_gradient_boosting_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Baseline: {np.mean(score_baseline_r2):.2f} (+/- {np.std(score_baseline_r2):.2f})")
# We also evaluate the baseline model to see how much better our models are compared to just guessing the average delay.
print(f"Mean Absolute Error (MAE) : Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
print(f"Mean Squared Error (MSE) : Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
print(f"R² Score : Baseline: {r2_score(y_test, y_pred_baseline):.2f}")

## Conclusion on the model comparison
For a first iteration, the results are pretty bad especially for the linear regression with an R² of -0.83 (+/- 0.57) which if far worst than the baseline.
For the Random Forest and the Gradient boosting are ok, they re not bad but they re not good either. with the random forest having a R² of 0.24 (+/- 0.16) and the Gradient boosting having a R² of 0.23 (+/- 0.16).
We can observe that the random forest is very slightly better than the Gradient boosting but only 0.01 witch make it pretty forgettable.

In [ ]:
# We create a global figure containing 1 row and 3 columns
# figsize=(18, 5) allows to have a wide image so that the 3 graphs fit well without being too small
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1 : Gradient Boosting (Case 0)
axes[0].scatter(y_test, y_pred_gradient_boosting, alpha=0.5)
axes[0].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[0].set_title("Real vs Predicted (Gradient Boosting)")
axes[0].set_xlabel("Real Delay")
axes[0].set_ylabel("Predicted Delay")

# Graph 2 : Random Forest (Case 1)
axes[1].scatter(y_test, y_pred_forest, alpha=0.5, color='green')
axes[1].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[1].set_title("Real vs Predicted (Random Forest)")
axes[1].set_xlabel("Real Delay")
axes[1].set_ylabel("Predicted Delay")

# Graph 3 : Linear Regression (Case 2)
axes[2].scatter(y_test, y_pred_lin, alpha=0.5, color='orange')
axes[2].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[2].set_title("Real vs Predicted (Linear Regression)")
axes[2].set_xlabel("Real Delay")
axes[2].set_ylabel("Predicted Delay")

# Adjust automatically the margins so that the titles and axes do not overlap
plt.tight_layout()

# Show all the graphs at once
plt.show()

# Second Model
We saw before that our result aren't very good which is normal because we didn t gave our model a lot of data to train on.
With what we did during the cleaning and the analyzing of our datas we can deduce new columns of data to give.
- Number of real trains : The actual number of trains that completed this journey during the month, excluding the canceled ones
- Number of trains delayed > 15min : The total number of trains that arrived at their destination more than 15 minutes behind schedule.
- Delay previous month : The average delay time this exact same route experienced during the previous month.

## Step 1: Adding new parameters
We add 3 new parameters in the features : Number of real trains, number of trains delayed > 15min & Delay previous month.
This mean that each model will have brand new data to trains to improve their accuracy.

In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of real trains', 'Number of trains delayed > 15min', 'Delay previous month']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 2 : Encoding

We encode non numerical values to usable by our models

In [ ]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=False is used to return a dense array instead of a sparse matrix, which is easier to handle in pipelines.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 3: Creating the AI "Assembly Line"

We setup each pipelines

In [ ]:
# We create a "Pipeline": it is an automated assembly line.
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

gradient_boosting_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

## Step 4: Creating the final AI

We finally do the training of each models

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
forest_model.fit(X_train, y_train)
linear_model.fit(X_train, y_train)
gradient_boosting_model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_forest_pred = forest_model.predict(X_test)
y_linear_pred = linear_model.predict(X_test)
y_gradient_boosting_pred = gradient_boosting_model.predict(X_test)
# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 5: Grading (Performance Evaluation)

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : Linear Regression: {mean_absolute_error(y_test, y_linear_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Forest Regression: {mean_absolute_error(y_test, y_forest_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Gradient Boosting: {mean_absolute_error(y_test, y_gradient_boosting_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : Linear Regression: {mean_squared_error(y_test, y_linear_pred):.2f}")
print(f"Mean Squared Error (MSE) : Forest Regression: {mean_squared_error(y_test, y_forest_pred):.2f}")
print(f"Mean Squared Error (MSE) : Gradient Boosting: {mean_squared_error(y_test, y_gradient_boosting_pred):.2f}")
print(f"Mean Squared Error (MSE) : Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : Linear Regression: {r2_score(y_test, y_linear_pred):.2f}")
print(f"R² Score : Forest Regression: {r2_score(y_test, y_forest_pred):.2f}")
print(f"R² Score : Gradient Boosting: {r2_score(y_test, y_gradient_boosting_pred):.2f}")
print(f"R² Score : Ba--- Graphique 3 : Linear Regression (Case 2) ---seline: {r2_score(y_test, y_pred_baseline):.2f}")
# We use cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_linear_r2 = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_forest_r2 = cross_val_score(forest_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_gradient_boosting_r2 = cross_val_score(gradient_boosting_model, X, y, cv=5, scoring='r2', n_jobs=-1)
score_baseline_r2 = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"\nR² average with Cross-Validation 5 : Linear Regression: {np.mean(scores_linear_r2):.2f} (+/- {np.std(scores_linear_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Forest Regression: {np.mean(scores_forest_r2):.2f} (+/- {np.std(scores_forest_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Gradient Boosting: {np.mean(scores_gradient_boosting_r2):.2f} (+/- {np.std(scores_gradient_boosting_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Baseline: {np.mean(score_baseline_r2):.2f} (+/- {np.std(score_baseline_r2):.2f})")

## Conclusion
We can see that adding new parameters were a success. The accuracy is far superior than before.
 - The linear model has a R² of 0.37 (+/- 0.12) meaning this time it is actually better than the baseline of -0.08 (+/- 0.09), but it s still not the best model.
 - The random forest has a R² of 0.76 (+/- 0.03) meaning that the model did actually improve pretty well and making it pretty accurate
 - The Gradient boosting has a R² of 0.78 (+/- 0.03) making it slightly better than the random forest.

In [ ]:
# Creation of a global figure containing 1 row and 3 columns
# figsize=(18, 5) allows to have a wide image so that the 3 graphs fit well without being too small
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1 : Gradient Boosting (Case 0)
axes[0].scatter(y_test, y_gradient_boosting_pred, alpha=0.5)
axes[0].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[0].set_title("Real vs Predicted (Gradient Boosting)")
axes[0].set_xlabel("Real Delay")
axes[0].set_ylabel("Predicted Delay")

# Graph 2 : Random Forest (Case 1)
axes[1].scatter(y_test, y_forest_pred, alpha=0.5, color='green')
axes[1].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[1].set_title("Real vs Predicted (Random Forest)")
axes[1].set_xlabel("Real Delay")
axes[1].set_ylabel("Predicted Delay")

# Graph 3 : Linear Regression (Case 2)
axes[2].scatter(y_test, y_linear_pred, alpha=0.5, color='orange')
axes[2].plot([0, y_test.max()], [0, y_test.max()], color='red', linewidth=2)
axes[2].set_title("Real vs Predicted (Linear Regression)")
axes[2].set_xlabel("Real Delay")
axes[2].set_ylabel("Predicted Delay")

# Adjust automatically the margins so that the titles and axes do not overlap
plt.tight_layout()

# Show all the graphs at once
plt.show()

# Hyperparameters
One solution to improve the model could be the Hyperparameters. Hyperparameters are parameters that guide how the model must learn. In our case we will use Grid Search. Grid Search is an algorythm that will try every combination of hyperparameters to search the best one.
Note : gridsearch can take a VERY long time if we input to much hyperparameters. We need to go by small batches and adjust each time to find the best combination.

## Step 1 : Setting up the data
We set up everything the same way that before

In [ ]:
target = 'Average delay of all trains at arrival'
#We go back to previous features because we saw that they didn't improve the model
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of real trains', 'Number of trains delayed > 15min', 'Delay previous month']

# We split our table into ts to guess (target)
X = df[features]
y = df[target]

# We create a "Pipeline": it is an automated assembly line.
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X contains only the clues (features)
# y contains only the answer


## Step 2 : We setup the grid search to try differents parameters
Same as before we to process the data before using.
We need to initiate which parameters the grid search will combines for each models, for that, we use a *grid*

In [ ]:
# We split the cols into categorical and numeric columns
categorical_cols = ['Service', 'Departure station', 'Arrival station', 'Season']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# We transform the categoricals values into numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# We create 3 pipelines for each models that first transforms the data and then applies the model
forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

gradient_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', HistGradientBoostingRegressor(random_state=42))
])

# We define the grid of hyperparameters to search over
forest_param_grid = {
    # We search for the best number of trees in the random forest
    'model__n_estimators': [100, 200, 300],
    # We search for the best maximum depth of the trees in the random forest
    'model__max_depth': [None],
    # We search for the best minimum number of samples required to split an internal node
    'model__min_samples_split': [3, 5, 7],
    # We search for the best minimum number of samples required to be at a leaf node
    'model__min_samples_leaf': [3, 4, 5],
    # We search for max features to consider when looking for the best split
    'model__max_features': [None]
}

linear_param_grid = {
    # We search for the best fit_intercept parameter for the linear regression
    'model__fit_intercept': [True, False],
    # We search for the best copy_X parameter for the linear regression
    'model__copy_X': [True, False],
}

gradient_param_grid = {
    # We search for the best number of boosting iterations (trees) in the gradient boosting model
    'model__max_iter': [450, 500, 550],
    # We search for the best maximum number of leaf nodes in the gradient boosting model
    'model__max_leaf_nodes': [17, 20, 22],
    # We search for the best learning rate in the gradient boosting model
    'model__learning_rate': [0.025, 0.05, 0.075],
    
    'model__min_samples_leaf': [10, 20, 30],

    'model__l2_regularization': [3.0, 4.0, 4.5]
}

# We create the Gridsearch algorythm that will search for the best hyperparameters
forest_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=forest_pipeline, 
    # The dictionary containing the hyperparameters to test
    param_grid=forest_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
    # Uses all available CPU cores to run calculations in parallel
    n_jobs=-1
)

linear_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=linear_pipeline,
    # The dictionary containing the hyperparameters to test
    param_grid=linear_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
     # Uses all available CPU cores to run calculations in parallel
     n_jobs=-1
)

gradient_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=gradient_pipeline,
    # The dictionary containing the hyperparameters to test
    param_grid=gradient_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
     # Uses all available CPU cores to run calculations in parallel
     n_jobs=-1
)

# We split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 3 : Executing the grid search

Now that everything is setup, we can execute it and see print the results of the grid search
- Warning :  This step can take a LOT of time and is ressources heavy for the computer.

In [ ]:
# We fit the grid search to the training data
linear_grid_search.fit(X_train, y_train)
forest_grid_search.fit(X_train, y_train)
gradient_grid_search.fit(X_train, y_train)

# We print the best hyperparameters found by the grid search
print("Best linear hyperparameters :", linear_grid_search.best_params_)
print("Best random forest hyperparameters :", forest_grid_search.best_params_)
print("Best gradient boosting hyperparameters :", gradient_grid_search.best_params_)

By executing the previous function multiple time, we got to the conclusion that the best parameters for each models are
- Linear : copy_X = True & fit_intercept = True
- Random Forest : max_depth = None, max_features = None, min_samples_leaf = 3, min_samples_split = 3 & n_estimators = 200
- Gradient Boosting : l2_regularization = 4.0, learning_rate = 0.05, max_iter = 500, max_leaf_nodes = 20 & min_samples_leaf = 20
We can now create a new model where we apply the hyperparameters for each models and see if it's better.

# Model 3

## Step 1 : initialization of the model
First we init the model like we did before


In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of real trains', 'Number of trains delayed > 15min', 'Delay previous month']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

# We create a "Pipeline": it is an automated assembly line. for the linear regression model
linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

# We create a "Pipeline": it is an automated assembly line for the random forest model.
model_forest = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# We create a "Pipeline": it is an automated assembly line for the gradient boosting model.
model_gradient_boosting = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])



## Step 2 : Apply hyperparameters
Now that we did the setup we can apply the hyperparameters we found for each models

In [ ]:
# Apply hyperparameter we found with GridSearchCV
# for the random forest model
model_forest.set_params(regressor__n_estimators=200)
model_forest.set_params(regressor__max_depth=None)
model_forest.set_params(regressor__min_samples_split=3)
model_forest.set_params(regressor__min_samples_leaf=3)
model_forest.set_params(regressor__max_features=None)
# for the linear regression model
linear_model.set_params(regressor__fit_intercept=True)
linear_model.set_params(regressor__copy_X=True)
# for the gradient boosting model
model_gradient_boosting.set_params(regressor__max_iter=500)
model_gradient_boosting.set_params(regressor__max_leaf_nodes=20)
model_gradient_boosting.set_params(regressor__learning_rate=0.05)
model_gradient_boosting.set_params(regressor__l2_regularization=4.0)
model_gradient_boosting.set_params(regressor__min_samples_leaf=20)


## Step 3 : Finishing the Ai
We can finally finish the Ai and test it after to see if the hyperparametters were an improvement or not

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model_forest.fit(X_train, y_train)
linear_model.fit(X_train, y_train)
model_gradient_boosting.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred_forest = model_forest.predict(X_test)
y_pred_linear = linear_model.predict(X_test)
y_pred_gradient_boosting = model_gradient_boosting.predict(X_test)
# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 4 : Grading the Ai
We can now grade the model to see if it improved

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) Random Forest: {mean_absolute_error(y_test, y_pred_forest):.2f} minutes")
print(f"Mean Absolute Error (MAE) Linear Regression: {mean_absolute_error(y_test, y_pred_linear):.2f} minutes")
print(f"Mean Absolute Error (MAE) Gradient Boosting: {mean_absolute_error(y_test, y_pred_gradient_boosting):.2f} minutes")
print(f"Mean Absolute Error (MAE) Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) Random Forest: {mean_squared_error(y_test, y_pred_forest):.2f}")
print(f"Mean Squared Error (MSE) Linear Regression: {mean_squared_error(y_test, y_pred_linear):.2f}")
print(f"Mean Squared Error (MSE) Gradient Boosting: {mean_squared_error(y_test, y_pred_gradient_boosting):.2f}")
print(f"Mean Squared Error (MSE) Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score Random Forest: {r2_score(y_test, y_pred_forest):.2f}")
print(f"R² Score Linear Regression: {r2_score(y_test, y_pred_linear):.2f}")
print(f"R² Score Gradient Boosting: {r2_score(y_test, y_pred_gradient_boosting):.2f}")
print(f"R² Score Baseline: {r2_score(y_test, y_pred_baseline):.2f}")
# We use cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_m3_forest = cross_val_score(model_forest, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_linear = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_gradient_boosting = cross_val_score(model_gradient_boosting, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_baseline = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"\nR² average with Cross-Validation 5 Random Forest: {np.mean(scores_m3_forest):.2f} (+/- {np.std(scores_m3_forest):.2f})")
print(f"\nR² average with Cross-Validation 5 Linear Regression: {np.mean(scores_m3_linear):.2f} (+/- {np.std(scores_m3_linear):.2f})")
print(f"\nR² average with Cross-Validation 5 Gradient Boosting: {np.mean(scores_m3_gradient_boosting):.2f} (+/- {np.std(scores_m3_gradient_boosting):.2f})")
print(f"\nR² average with Cross-Validation 5 Baseline: {np.mean(scores_m3_baseline):.2f} (+/- {np.std(scores_m3_baseline):.2f})")


We can see that the hyperparametters improved only the gradient boosting but very slightly and the others models did not change. It was expected because the default hyperparameters are studied on multiple dataset to be the most efficient
Before / After hyperparameters
- Linear : 0.37 (+/- 0.12) | 0.37 (+/- 0.12)
- Random forest : 0.76 (+/- 0.03) | 0.76 (+/- 0.03)
- Gradient boosting : 0.78 (+/- 0.03) | 0.79 (+/- 0.03)

In [ ]:
errors = y_test - y_pred_gradient_boosting

# We draw a histogram to visualize the distribution of errors (residuals) for the gradient boosting model.
errors.plot(kind='hist', bins=50, figsize=(8, 5), color='purple', edgecolor='white')

# We add titles and labels to the histogram to make it easier to understand.
plt.title("Errors distribution (Gradient Boosting)")
plt.xlabel("Error in minutes (Negative = Overestimated, Positive = Underestimated)")
plt.ylabel("Number of predictions")
plt.show()

# Data influence
We can see which datas influenced the most in the prediction, this information can be good to show on the dashboard after

In [ ]:
resultats_importance = permutation_importance(
    model_gradient_boosting, X_test, y_test, 
    n_repeats=10, random_state=42, n_jobs=-1
)

# Recuperation of the importance values and the corresponding feature names
importances_moyennes = resultats_importance.importances_mean
noms_features = X_test.columns

# Sorting the features by importance in descending order
index_tries = importances_moyennes.argsort()[::-1]
importances_triees = importances_moyennes[index_tries]
features_triees = noms_features[index_tries]
# We visualize the importance of the features using a horizontal bar chart with Seaborn
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))
# We create a horizontal bar chart where:
ax = sns.barplot(
    x=importances_triees, 
    y=features_triees, 
)
# Add the importance values at the end of each bar for better readability
for i, v in enumerate(importances_triees):
    ax.text(v + 0.005, i, f"{v:.3f}", color='black', va='center', fontweight='bold')
# We add titles and labels to the graph to make it easier to understand.
plt.title("Top des informations les plus influentes sur le retard des trains", fontsize=16, pad=20, fontweight='bold')
plt.xlabel("Niveau d'impact (Baisse de performance si la donnée est retirée)", fontsize=12)
plt.ylabel("Données", fontsize=12)
plt.tight_layout()
plt.show()

# Conclusion
After every training we can see that both random forest and gradient boosting are far superior than linear regression. The choice is now only between Random forest and gradient boosting but after hyperparameters tunning, we can see that the gradient boosting is slightly superior making it our choice for our dashboard.
We now only have to export it to be usable in the dashboard.

In [ ]:
# We export the best model (gradient boosting) to reuse it later without having to retrain it.
joblib.dump(model_gradient_boosting, 'model.joblib')
print("Successfully exported gradient boosting model as 'model.joblib'")